# Compare Students delivery type

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/compare_students.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Load Git Repository
# @markdown Clone the Git repository for analysis if needed.
clone_git_repo = False  # @param {type:"boolean"}
if clone_git_repo:
    !git clone https://github.com/PeaceAndLongLife/Analysis-Colab.git
    %cd Analysis-Colab


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("src").resolve()))
%cd notebooks

In [ ]:
# @title Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FIL
from google.colab import userdata

# SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')
SERVICE_ACCOUNT_FILE = '/content/drive/Shareddrives/Travis Research Drive/Interface 3.0/Admin files/service_account.json'

# @title ## Install pyDrive  {"form-width":"20%"}

# @markdown ---
# @markdown Installing PyDrive
# @markdown
# @markdown Thie is necessary if using the Google API to call files by their file_id

# !pip install PyDrive

import sys
# 2. Tell Python to look in your custom folder
# sys.path.append(PACKAGE_PATH)
sys.path.append('src')
# 3. Import your file!
from GoogleFunctions import extract_file_id
from local_io import read_csv_from_id


## Read in data


In [ ]:
# @ title ## Input Data Links  {"form-width":"20%"}
# @markdown ---
# Fall data info
# @markdown Read in fall online file {"form-width":"20%"}
fall_online_link = "https://drive.google.com/file/d/1_Gb6kUjWEWFt3pqlI3JJvExPfyvqUB8u/view?usp=drive_link" # @param {"type":"string"}
fall_online_link_id = extract_file_id(fall_online_link)
show_fall_online = False # @param {"type":"boolean"}

# @markdown Read in fall in person file {"form-width":"20%"}
fall_inperson_link = "https://drive.google.com/file/d/12c-U4mECjr3xmjTHNBzbPEYYY5SYM2Cj/view?usp=drive_link" # @param {"type":"string"}
fall_inperson_link_id = extract_file_id(fall_inperson_link)
show_fall_inperson = False # @param {"type":"boolean"}

fall_online_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, fall_online_link_id, show_fall_online)
fall_inperson_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, fall_inperson_link_id, show_fall_inperson)


# @markdown ---
# Winter data info
# @markdown Read in winter online file {"form-width":"20%"}
winter_online_link = "https://drive.google.com/file/d/14zR09W0Tr_lL_wbjAXjtHeWcrmGxAnDd/view?usp=drive_link" # @param {"type":"string"}
winter_online_link_id = extract_file_id(winter_online_link)
show_winter_online = False # @param {"type":"boolean"}

# @markdown Read in fall in person file {"form-width":"20%"}
winter_inperson_link = "https://drive.google.com/file/d/1i4j2ARL1x-FOvDz5gA6gJmxWtT2bHfgk/view?usp=drive_link" # @param {"type":"string"}
winter_inperson_link_id = extract_file_id(winter_inperson_link)
show_winter_inperson = False # @param {"type":"boolean"}

winter_online_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, winter_online_link_id, show_winter_online)
winter_inperson_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, winter_inperson_link_id, show_winter_inperson)




In [ ]:
# @markdown Read in fall in person file {"form-width":"20%"}
userprofile_link = "https://drive.google.com/file/d/1k42GKGxllb97GL7J9q_IFAS0cQ5J4p-c/view?usp=drive_link" # @param {"type":"string"}
userprofile_link_id = extract_file_id(userprofile_link)
show_userprofile = True # @param {"type":"boolean"}

userprofile_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, userprofile_link_id, show_userprofile)
userprofile_df = userprofile_df.rename(columns={'username': 'user'})
user_map = userprofile_df.set_index('user')['id'].squeeze()

In [ ]:
display(user_map)

In [ ]:
def strip_id(df):
    if 'login_id' in df.columns:
        df['user'] = df['login_id'] + '@pdx.edu'
        # df = df.drop(df.iloc[[0, 1]].index)
        df['user'] = df['user'].map(user_map).fillna(pd.NA).astype('Int64')
        # df.set_index('user', inplace=True)
    return df
fall_online_stripped_df = strip_id(fall_online_df)
fall_online_stripped_df = fall_online_stripped_df.drop(columns=['name', 'login_id', 'canvas_user_id'])
display(fall_online_stripped_df)

fall_inperson_stripped_df = strip_id(fall_inperson_df)  
fall_inperson_stripped_df = fall_inperson_stripped_df.drop(columns=['name', 'login_id', 'canvas_user_id']) 
display(fall_inperson_stripped_df)

winter_online_stripped_df = strip_id(winter_online_df)
winter_online_stripped_df = winter_online_stripped_df.drop(columns=['name', 'login_id', 'canvas_user_id'])
display(winter_online_stripped_df)

winter_inperson_stripped_df = strip_id(winter_inperson_df)
winter_inperson_stripped_df = winter_inperson_stripped_df.drop(columns=['name', 'login_id', 'canvas_user_id'])
display(winter_inperson_stripped_df)


In [ ]:
import pandas as pd


def compare_users_in_df(df1,df2,col):


    # display(df1,df2)
    # This returns rows from df1 where the name is also present in df2
    common_names = df1[df1[col].isin(df2[col])]

    print("\nRows in that match:")
    display(common_names)

print("fall_online to winter_inperson")
compare_users_in_df(winter_inperson_stripped_df,fall_online_stripped_df,'user')
print("\n---\n")

print("fall_inperson to winter_online")
compare_users_in_df(winter_online_stripped_df,fall_inperson_stripped_df,'user')
print("\n---\n")